Data Analyst: Andrei Berner Arroyo Tanta

**Project with Numpy 2: E-Commerce Price Intelligence & Market Sentiment Engine

**Business Application:
In the highly competitive e-commerce sector, this engine serves as a safeguard against price anomalies and a tool for auditing marketing integrity during peak sales periods.

-Business Need: Detect unusual price fluctuations, identify "fake discounts" during Black Friday, and understand product relationships.

-Area of Benefit: Marketing Strategy, Pricing Compliance and Consumer Trust.

-Decision Support: Identifies optimal product bundles based on correlation and protects brand reputation by ensuring promotional transparency.

**Executive Summary

-Anomaly Detection: The system triggered 2 critical alerts for Smartphone 1 after detecting deviations exceeding 5% from the historical average price.

-Black Friday Integrity Audit: A rigorous comparison against the year-to-date average revealed that only 1 out of 7 products (Power Bank 1) offered a genuine 15%+ discount. The remaining products showed no significant price reduction, providing evidence for a "fake discount" analysis.

-Volatility Analysis: Data segmentation showed that Q4 volatility is 226.31% higher than Q3, confirming the intense price wars characteristic of the holiday season.

-Market Correlation: Discovered a high positive correlation of 0.89 between Smartphone 1 and Airpods 1, validating their status as complementary goods for bundling strategies.

In [5]:
#Although this project is about NumPy, we will load pandas exclusively for the operation of loading data from a dataset.
import numpy as np
import pandas as pd

In [6]:
#Loadin raw data for processing
dataset_path = "product_tracks.xls"
df = pd.read_csv(dataset_path)

#Memory Optimization: Pre-allocating ndarrays
#We use float32 for prices to reduce memory footprint vs float64
num_rows = 184
prices_report = np.empty((num_rows, 7), dtype=np.float32)
date_report = np.empty((num_rows, 1), dtype="datetime64[D]")

#Transferring data into the optimized NumPy engine
prices_report[:] = df.iloc[:, 1:].to_numpy()
date_report[:] = df.iloc[:, :1].to_numpy(dtype="datetime64[D]")

np.set_printoptions(suppress=True, precision=2)
print(f"Engine Ready. Memory allocated for {prices_report.nbytes} bytes of price data.")

Engine Ready. Memory allocated for 5152 bytes of price data.


In [7]:
#Multi-Level Anomaly Detection (Boolean Indexing)
#Calculating global statistics across the 6-month period
average_prices = np.mean(prices_report, axis=0)
thresholds = average_prices * 1.05

#Targeted Analysis: Smartphone 1 (Index 0)
#Demonstrating Boolean Masking for rapid filtering
product_idx = 0 #for the following products only change this value
anomaly_mask = prices_report[:, product_idx] > thresholds[product_idx]

#Extracting specific dates and values using the mask
anomaly_dates = date_report[anomaly_mask].astype(str)
anomaly_values = prices_report[anomaly_mask, product_idx]

#Consolidating results for reporting
print(f"--- Report ---")
anomaly_report = np.column_stack((anomaly_dates, anomaly_values))
print(f"Alerts Triggered for Product {product_idx}: {len(anomaly_report)}")
print("Sample Anomalies (Date and Price):\n", anomaly_report[:5])

--- Report ---
Alerts Triggered for Product 0: 2
Sample Anomalies (Date and Price):
 [['2025-09-29' '661.1']
 ['2025-10-07' '660.2']]


In [8]:
#Temporal Segmentation
#Locate the transition to Q4 (October 1st)
q4_start_idx = np.where(date_report[:, 0] == np.datetime64("2025-10-01"))[0][0]

#Creating VIEWS for Quarter 3 and Quarter 4
#Changes to these slices would reflect in 'prices_report'
prices_q3 = prices_report[:q4_start_idx, :]
prices_q4 = prices_report[q4_start_idx:, :]

#Comparing Volatility between Quarters
vol_q3 = np.mean(np.abs(np.diff(prices_q3, axis=0)))
vol_q4 = np.mean(np.abs(np.diff(prices_q4, axis=0)))
vol_pct= np.round(((vol_q4/vol_q3)*100),2)

print(f"--- Report ---")
print(f"Q3 Volatility: {vol_q3:.2f}")
print(f"Q4 Volatility: {vol_q4:.2f}")
print(f"Insight: Q4 shows significantly higher volatility due to holiday sales events of", vol_pct, f"%." )

--- Report ---
Q3 Volatility: 2.55
Q4 Volatility: 5.77
Insight: Q4 shows significantly higher volatility due to holiday sales events of 226.31 %.


In [9]:
#Vectorized Volatility & Day-to-Day Trends
#Vectorized calculation of daily percentage change
daily_diff = np.diff(prices_report, axis=0)
previous_day_prices = prices_report[:-1, :]
daily_pct_change = (daily_diff / previous_day_prices) * 100

# Identify significant daily shifts (>5% move)
extreme_shift_mask = np.abs(daily_pct_change) > 5
shift_counts = np.sum(extreme_shift_mask, axis=0)

print(f"--- Report ---")
print(f"Significant Price Shifts detected per product:")
print(shift_counts)

--- Report ---
Significant Price Shifts detected per product:
[ 8  2  6  4  3 12  2]


Our system has identified 2 alerts for Product 0 (Smartphone 1), signaling unusual price movements that deviate from expected patterns, 5% of avarage price on the following dates and prices:
date 1: 2025-09-29, price: 661.1
date 2: 2025-10-07, price: 660.2

In the analysis of quarterly volatility metrics, this reveals clear behavioral differences:
Q3 Volatility: 2.55
Q4 Volatility: 5.77 
Q4 exhibits 226.31% higher volatility, driven by holiday sales cycles, aggressive discounting, and competitive pricing dynamics. This reinforces the need for automated monitoring during peak retail seasons.

And in the analysis of daily price volatility for each product, we see that the product with the highest volatility is the fifth product, while the second and the last products are the most stable.

In [10]:
#Event-Driven Audit: Black Friday "Fake Discounts"
#Find Black Friday (bf) Index (Nov 28, 2025)
bf_date = np.datetime64("2025-11-28")
bf_idx = np.where(date_report[:, 0] == bf_date)[0][0]

#Is the BF Price < 85% of the average price of the year so far?
avg_price_pre_bf = np.mean(prices_report[:bf_idx], axis=0)
bf_prices = prices_report[bf_idx]

is_real_offer = bf_prices < (avg_price_pre_bf * 0.85)

print(f"--- Report ---")
#SM stands for Smartphone, Air for Airphones and PB for Power Bank. These are the products of our analysis
print(f"Black Friday Audit (Real 15%+ Discount vs Year Average):")
products = ["SM1", "SM2", "Air1", "Air2", "Air3", "PB1", "PB2"]
for i, status in enumerate(is_real_offer):
    print(f"{products[i]}: {'REAL OFFER' if status else 'NO REAL OFFER'}")

--- Report ---
Black Friday Audit (Real 15%+ Discount vs Year Average):
SM1: NO REAL OFFER
SM2: NO REAL OFFER
Air1: NO REAL OFFER
Air2: NO REAL OFFER
Air3: NO REAL OFFER
PB1: REAL OFFER
PB2: NO REAL OFFER


In [11]:
#Market Relationship Analysis (Correlation)
#Correlation between Smartphone 1 and Airpods 1 (Complementary Goods)
sm1_prices = prices_report[q4_start_idx:, 0]
air1_prices = prices_report[q4_start_idx:, 2]

correlation = np.corrcoef(sm1_prices, air1_prices)[0, 1]

print(f"--- Report ---")
print(f"Correlation (Phone 1 & Airpods 1): {correlation:.2f}")
# Result: High positive correlation suggests these are sold/discounted as a bundle.

--- Report ---
Correlation (Phone 1 & Airpods 1): 0.89


In [13]:
#Compliance & Holiday Auditing (Fancy Indexing)
#Identify Christmas Index
xmas_idx = np.where(date_report[:, 0] == np.datetime64("2025-12-25"))[0][0]

#Practiving Fancy Indexing: Selecting specific non-contiguous products (Smartphone1, Airpod1, Powerbank1)
#Christmas index
compliance_check_indices = [0, 2, 5]
holiday_budget_prices = prices_report[xmas_idx, compliance_check_indices]

print(f"--- Report ---")
budget_total = np.sum(holiday_budget_prices)
print(f"Christmas 'Essential Budget' Price: €{budget_total:.2f}")

--- Report ---
Christmas 'Essential Budget' Price: €680.85


To evaluate discount authenticity during Black Friday (Nov 28, 2025), the system compared each product’s Black Friday price against 85% of its year‑to‑date average price.
Black Friday Audit — Real 15%+ Discounts
Only Power Bank 1 (PB1) qualifies as a genuine 15%+ discount, while the remaining products show no evidence of meaningful price reductions—an important insight for detecting for “fake discount”

The system also identified a strong positive correlation between:
Smartphone 1 and the Airpods 1 (Correlation: 0.89)
This suggests complementary purchasing behavior, meaning price changes in one product may influence demand for the other.

Finally, using fancy indexing, the system evaluated essential product prices on December 25, 2025, focusing on:
Smartphone 1, Airpod 1 and Power Bank 1
The combined Christmas “Essential Budget” totals is €680.85
This metric supports holiday pricing compliance and helps assess affordability during peak consumer periods.